# II2202 SNN-vs-ANN: pilot run on Colab

Research plan **Step 6 (pilot)**: one seed, full N-MNIST, both arms. It answers four questions:
1. Does everything run on the GPU, and how long and how much memory does one seed take?
2. **Recheck of the conversion gate** with a fully trained ANN (decision D-B).
3. Does the accuracy-vs-T curve depend on how the T-step input is formed? (P1 vs P2, `docs/PRIMER.md` §5)
4. Do both arms learn, and does the T grid separate them?

One seed is **not** a result for the research question; it fixes the protocol (plan Step 7).
Full explanations: `docs/COLAB_PILOT_WALKTHROUGH.md`.

**Before running:** *Runtime → Change runtime type → T4 GPU*. Run the cells top to bottom.
After a disconnect, re-run cells 1–4 (they are safe to repeat). The data must be downloaded again in every new session.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Storage: results go to Google Drive (survives disconnects); data stays on local disk (fast)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS = '/content/drive/MyDrive/snn_results'   # all outputs and logs
DATA    = '/content/data'                         # N-MNIST (local disk; Drive is too slow for 70k small files)
REPO    = '/content/SNN-vs-ANN'
BRANCH  = 'feat/event-based-temporal-pipeline'
!mkdir -p {RESULTS}

## 3. Code: clone, or update if already cloned. Prints the exact commit that runs

In [ ]:
import os
if os.path.isdir(REPO):
    !git -C {REPO} fetch -q origin && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} https://github.com/liviacardaccia02/SNN-vs-ANN.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q tonic==1.7.0 spikingjelly==0.0.0.0.14 pytest

In [ ]:
# Record the environment (copy this into your run log)
import sys, torch, importlib.metadata as md
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda,
      '| gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('spikingjelly', md.version('spikingjelly'), '| tonic', md.version('tonic'))

## 4. Fast tests (no dataset needed, ~3–4 min). Optional on re-runs
Includes the regression tests for the two bugs from the first Colab attempt
(percentile above 16 M values; output paths on Drive).

In [ ]:
!python -m pytest tests/test_temporal_transforms.py tests/test_models.py tests/test_conversion.py -q -p no:warnings

## 5. Dataset (~1.1 GB, ~2–10 min; skipped if already present)

In [ ]:
import os
if not (os.path.isdir(f'{DATA}/NMNIST/Train') and os.path.isdir(f'{DATA}/NMNIST/Test')):
    !python -c "import tonic; tonic.datasets.NMNIST('{DATA}', train=False); tonic.datasets.NMNIST('{DATA}', train=True)"
else:
    print('N-MNIST already on disk')

## 6. Pipeline smoke test on real data (~2 min)
Expected last line: `STATUS: PASS (28 checks, 0 critical failures, ...)` and `device_used: cuda`.

In [ ]:
!python scripts/test_event_pipeline.py --data-root {DATA} --device cuda --out-dir {RESULTS}/smoke 2>&1 | tail -n 25

## 7. Timing probe: 1 epoch of every stage, full data (~20–40 min)
Measures seconds per epoch and memory *before* committing hours. The gate result here means nothing
(the ANN has trained only 1 epoch), so the gate is set to `report` for this probe only.

In [ ]:
!python scripts/run_pilot.py --seed 0 --data-root {DATA} \
    --ann-epochs 1 --ft-epochs 1 --sg-epochs 1 --no-sensitivity --gate-mode report \
    --output-dir {RESULTS}/timing_probe 2>&1 | tee {RESULTS}/timing_probe.log

In [ ]:
# Estimate the full pilot from the probe: max epochs x seconds/epoch (an upper bound; early stopping usually ends sooner)
import re, json
log = open(f'{RESULTS}/timing_probe.log').read()
sec = {k: float(m.group(1)) for k, pat in [('ann', r'\[ANN seed=00\] Epoch\s+1/1.*?time=([\d.]+)s'),
                                          ('ft',  r'\[SNN-FT seed=00\] Epoch\s+1/1.*?time=([\d.]+)s'),
                                          ('sg',  r'\[SNN-SG seed=00\] Epoch\s+1/1.*?time=([\d.]+)s')]
       if (m := re.search(pat, log))}
print('seconds per epoch:', sec)
mem = re.findall(r'Peak GPU memory \(Method ([AB])\): ([\d.]+|None) MB', log)
print('peak GPU memory (MB):', dict(mem))
if len(sec) == 3:
    est = 30*sec['ann'] + 10*sec['ft'] + 30*sec['sg']
    print(f'full pilot upper bound (30 ANN + 10 FT + 30 SG epochs, excl. evaluation): {est/3600:.1f} h')
    print('If this is above ~8 h, split the run or reduce epochs for the pilot only (and record it).')
else:
    print('Probe incomplete; read the end of the log:'); print(log[-2500:])

## 8. Full pilot, protocol P1 (train at T = 24, evaluate at T ∈ {3, 6, 12, 24, 48}, plus sensitivity variants)
Long-running. The staged conversion gate is active (D-B). If you see `CUDA out of memory`, add `--batch-size 64` and record it.

In [ ]:
!python scripts/run_pilot.py --seed 0 --data-root {DATA} \
    --output-dir {RESULTS}/pilot_seed00_Ttrain24 2>&1 | tee {RESULTS}/pilot_seed00_Ttrain24.log

In [ ]:
# Summary of the pilot (all numbers come from results.json)
import json
def summarize(path):
    r = json.load(open(path))
    b, a = r['method_b_hybrid'], r['method_a_sg']
    fid = b['training'].get('conversion_fidelity', {})
    print('ANN test acc {:.4f} -> target (threshold) {:.4f}'.format(r['ann_baseline']['test_acc'], r['ann_baseline']['threshold_s']))
    print('Gate recheck: outcome={} | strict={} sanity={} (>= {:.4f}) | converted SNN val {:.4f} vs ANN val {:.4f}, agreement {:.4f}'.format(
        fid.get('gate_outcome'), fid.get('passed'), fid.get('sanity_passed'), fid.get('sanity_threshold', float('nan')),
        fid.get('snn_acc', float('nan')), fid.get('ann_acc', float('nan')), fid.get('agreement', float('nan'))))
    print('\n  T   | SG acc  q | Hybrid acc q')
    sg = {e['T']: e for e in a['evaluation']['snn_evals']}; ft = {e['T']: e for e in b['evaluation']['snn_evals']}
    for T in sorted(sg):
        print(f"  {T:3d} | {sg[T]['test_acc']:.4f} {'*' if sg[T]['qualifies'] else ' '} | {ft[T]['test_acc']:.4f}    {'*' if ft[T]['qualifies'] else ' '}")
    print('min qualifying T: SG =', a['evaluation']['min_qualifying_T'], '| Hybrid =', b['evaluation']['min_qualifying_T'])
    print('\nTraining budget (sample-time-steps): SG {:,} | Hybrid {:,} (ANN {:,} + FT {:,})'.format(
        a['training']['train_sample_timesteps'], b['training']['train_sample_timesteps'],
        b['training']['ann_sample_timesteps'], b['training']['ft_sample_timesteps']))
    print('Wall time (s): SG {:.0f} | Hybrid {:.0f}   Peak GPU MB: SG {} | Hybrid {}'.format(
        a['training']['total_wall_s'], b['training']['total_wall_s'], a['training']['peak_gpu_mb'], b['training']['peak_gpu_mb']))
    for v, res in r.get('evaluation_sensitivity', {}).items():
        print(f"sensitivity [{v}]: min T  SG={res['snn_sg']['min_qualifying_T']}  Hybrid={res['snn_ft']['min_qualifying_T']}  | "
              + ' '.join(f"T{e['T']}:{e['test_acc']:.3f}/{f['test_acc']:.3f}" for e, f in zip(res['snn_sg']['snn_evals'], res['snn_ft']['snn_evals'])))
    print('\nPilot checks:'); [print(('  [PASS] ' if c['passed'] else '  [FAIL] ') + c['label']) for c in r['pilot_checks']]
    return r
p1 = summarize(f'{RESULTS}/pilot_seed00_Ttrain24/results.json')

## 9. P2 probe: train at T = 3 (is the T = 3 accuracy of the T = 24 model understated?)

In [ ]:
!python scripts/run_pilot.py --seed 0 --data-root {DATA} --t-train 3 --no-sensitivity \
    --output-dir {RESULTS}/pilot_seed00_Ttrain3 2>&1 | tee {RESULTS}/pilot_seed00_Ttrain3.log

In [ ]:
# Decision rule fixed before the pilot (docs/DECISIONS.md, D-A addendum)
p2 = json.load(open(f'{RESULTS}/pilot_seed00_Ttrain3/results.json'))
def acc_at(r, arm, T):
    key = 'method_a_sg' if arm == 'sg' else 'method_b_hybrid'
    return {e['T']: e['test_acc'] for e in r[key]['evaluation']['snn_evals']}[T]
for arm in ('sg', 'hybrid'):
    a1, a2 = acc_at(p1, arm, 3), acc_at(p2, arm, 3)
    print(f'{arm:6s}: accuracy at T=3  P1 (trained at 24) {a1:.4f} | P2 (trained at 3) {a2:.4f} | P2-P1 = {100*(a2-a1):+.1f} pp')
print('\nRule: P2 within ~1-2 pp of P1 AND sensitivity variants agree -> keep P1; otherwise use P2 for the main experiment.')

## 10. Record before the main experiment (plan Step 7)
Write into `docs/DECISIONS.md` (in the repo, via a commit):
1. Gate recheck outcome (strict and sanity numbers above).
2. P1 or P2 (cell 9).
3. Seed count and statistical test (from the wall time per seed).
4. Any deviations: batch size, reduced epochs, interrupted runs.

Keep the `.log` and `results.json` files on Drive; they are your evidence.